# MosekMonolithicSDP

`MosekMonolithicSDP` lifts a D=1 `QcqpProblem` into one positive-semidefinite matrix, solves it with the optional MOSEK Fusion backend, and recovers one QCQP vector per diagonal block.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415  
All Rights Reserved  
Authors: Frank Dellaert, et al. (see THANKS for the full author list)  
See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/certifiable/doc/MosekMonolithicSDP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam
from gtsam.symbol_shorthand import X

## Formulation and lifecycle

For a QCQP with stacked vector $x$, the relaxation replaces $xx^\top$ by one matrix $Z\succeq0$. Quadratic costs and constraints become linear trace expressions in $Z$. The Python lifecycle is:

1. Construct `MosekMonolithicSDP(problem)`.
2. Call `solve(mosek_params)`.
3. Inspect `objectiveValue`, `problemStatus`, and `solveTimeSeconds`.
4. Call the functional `qcqpValues()` query and convert its matrix-valued result with the typed QCQP extraction helpers.
5. Query `variableEVRs()` independently when rank-one diagnostics are needed.

In [3]:
num_poses = 3
step = gtsam.Pose2(2.0, 0.0, 2.0 * np.pi / num_poses)
ground_truth = [gtsam.Pose2()]
for _ in range(1, num_poses):
    ground_truth.append(ground_truth[-1].compose(step))

graph = gtsam.NonlinearFactorGraph()
graph.add(gtsam.FrobeniusPriorPose2(
    X(0), ground_truth[0].matrix(), gtsam.noiseModel.Constrained.All(9)
))
for i in range(num_poses):
    j = (i + 1) % num_poses
    graph.add(gtsam.FrobeniusBetweenFactorPose2(
        X(i), X(j), ground_truth[i].between(ground_truth[j])
    ))
problem = gtsam.QcqpProblem(graph)

In [4]:
if not hasattr(gtsam, "MosekMonolithicSDP"):
    print("This GTSAM build does not include the optional MOSEK backend.")
else:
    solver = gtsam.MosekMonolithicSDP(problem)
    if not solver.solve({"intpntCoTolRelGap": 1e-10}):
        raise RuntimeError("MOSEK did not return a readable primal solution")
    recovered = gtsam.extractQcqpValuesPose2(solver.qcqpValues())
    errors = [
        np.linalg.norm(ground_truth[i].localCoordinates(recovered.atPose2(X(i))))
        for i in range(num_poses)
    ]
    print("status:", solver.problemStatus())
    print("objective:", solver.objectiveValue())
    print("solve time (s):", solver.solveTimeSeconds())
    print("ordered keys:", solver.orderedKeys())
    print("ordered dimensions:", solver.orderedKeyDims())
    print("eigenvalue ratios:", solver.variableEVRs())
    print("maximum Pose2 error:", max(errors))

status: ProblemStatus::PrimalAndDualFeasible
objective: 1.460840337585978e-10
solve time (s): 0.012079000473022461
ordered keys: [8646911284551352320, 8646911284551352321, 8646911284551352322]
ordered dimensions: {8646911284551352320: 7, 8646911284551352321: 7, 8646911284551352322: 7}
eigenvalue ratios: [805108656053.8125, 136661485370.81117, 79782496289.33049]
maximum Pose2 error: 1.2024240357747173e-06


`variableEVRs()` reports the largest-to-second-largest eigenvalue ratio of each recovered diagonal block. A large ratio indicates a nearly rank-one block and makes vector recovery meaningful. This class is available in Python only when GTSAM is built with MOSEK.